# Neg Sampler Tutorial

This tutorial shows the minimal workflow for `pepbenchmark.neg_sampler`: build a pool, run sampling, and inspect strategies and results.

In [1]:
from pepbenchmark.neg_sampler import (
    NegSampler,
    SamplingPoolManager,
    StrategySelector,
    list_available_sampling_strategies,
)

print('Available strategies:', list_available_sampling_strategies())

/home/batchcom/assist/miniforge3/envs/pepbenchmark/lib/python3.10/site-packages/outdated/__init__.py:36: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import parse_version


Available strategies: ['bin', 'chunk_shuffle', 'hybrid_property_kmer', 'kde', 'klet_shuffle', 'mmd', 'moment', 'nn', 'ot', 'random']


## 1. Build a Simple Candidate Pool

In [ ]:
positive_sequences = [
    'ACDEFG',
    'KLMNPQ',
    'QRSTVW',
]

candidate_sequences = [
    'AAAAAA', 'CCCCCC', 'DDDDDD', 'EEEEEE', 'FFFFGG',
    'GGGGGG', 'HHHHHH', 'IIIKKK', 'LLLMMM', 'NNNPPP',
]

pool_manager = SamplingPoolManager(include_sequences=candidate_sequences)
pool_manager.filter_by_length(min_length=6, max_length=6)
print('Pool size:', pool_manager.get_pool_size())
print('Pool preview:', pool_manager.get_sampling_pool()[:5])

2026-03-27 05:21:10 | INFO     | pepbenchmark.neg_sampler.sampling_pool_manager | Added 10 user-provided sequences
2026-03-27 05:21:10 | INFO     | pepbenchmark.neg_sampler.sampling_pool_manager | Added 10 raw sequences. Pool size: 0 -> 10 (+10)
2026-03-27 05:21:10 | INFO     | pepbenchmark.neg_sampler.sampling_pool_manager | Length filter (min=6, max=6): 10 -> 10
Pool size: 10
Pool preview: ['FFFFGG', 'GGGGGG', 'EEEEEE', 'DDDDDD', 'LLLMMM']


## 2. Run Random Negative Sampling

In [3]:
sampler = NegSampler(pool_manager.get_sampling_pool(), positive_sequences)
negatives = sampler.sample_negatives(
    method='random',
    properties=['length', 'charge'],
    ratio=1.0,
    seed=42,
)

print('Positive count:', len(positive_sequences))
print('Negative count:', len(negatives))
print('Sampled negatives:', negatives)

2026-03-27 05:21:12 | INFO     | pepbenchmark | Available samplers: ['bin', 'chunk_shuffle', 'hybrid_property_kmer', 'kde', 'klet_shuffle', 'mmd', 'moment', 'nn', 'ot', 'random']
2026-03-27 05:21:12 | INFO     | pepbenchmark | [random] Final negatives=3 (initial=0, sampled=3)
Positive count: 3
Negative count: 3
Sampled negatives: ['CCCCCC', 'FFFFGG', 'NNNPPP']


## 3. Generate a Distribution-Difference Report

In [4]:
report_text, report_dict = sampler.generate_similarity_report(
    properties=['length', 'charge'],
    use_sampled_only=True,
)

print(report_text[:1200])
print('Report keys:', sorted(report_dict.keys()))

Distribution Similarity Report
Positive sequences: 3
Negative sequences: 3
Properties analyzed: 2
Properties similar: 1/2
Overall similar: ✗ No

Property Details:
--------------------
✓ length
✗ charge (JS divergence 0.8326 > 0.1; KS stat 0.6667 > 0.1)

Statistical Summary:
--------------------
length: KS p-val=1.0000, JS div=0.0000, Mean diff=0.0000
charge: KS p-val=0.6000, JS div=0.8326, Mean diff=0.0294
Report keys: ['distribution_comparison', 'similarity_summary', 'summary', 'thresholds']


## 4. Automatically Compare Multiple Strategies

In [5]:
selector = StrategySelector(prefer='condition_pass', verbose=False)
strategies = [
    {'method': 'random', 'params': {}},
    {'method': 'bin', 'params': {'n_bins': 5}},
    {'method': 'nn', 'params': {'k_per_pos': 1}},
]

best, summary_rows, summary_df, strategy_results = selector.run_and_select(
    sampler,
    properties=['length', 'charge'],
    strategies=strategies,
    ratio=1,
    seed=42,
    report_kwargs={'use_sampled_only': True},
    print_summary=False,
)

print('Best strategy:', None if best is None else best['method'])
print(summary_df[['method', 'success', 'mean_js', 'mean_ks']])

2026-03-27 05:21:19 | INFO     | pepbenchmark | Starting strategy selection with 3 strategies
2026-03-27 05:21:19 | INFO     | pepbenchmark | Properties: ['length', 'charge'], ratio: 1, seed: 42
2026-03-27 05:21:19 | INFO     | pepbenchmark | === Testing All Sampling Strategies ===
=== Testing All Sampling Strategies ===
2026-03-27 05:21:19 | INFO     | pepbenchmark | Testing RANDOM strategy with params: {}

--- Testing RANDOM strategy ---

2026-03-27 05:21:19 | INFO     | pepbenchmark | [random] Final negatives=3 (initial=0, sampled=3)
2026-03-27 05:21:19 | INFO     | pepbenchmark | ✓ RANDOM: Success - generated 3 negatives
2026-03-27 05:21:19 | INFO     | pepbenchmark | Testing BIN strategy with params: {'n_bins': 5}

--- Testing BIN strategy ---

2026-03-27 05:21:19 | INFO     | pepbenchmark | [bin] Final negatives=3 (initial=0, sampled=3)
2026-03-27 05:21:19 | INFO     | pepbenchmark | ✓ BIN: Success - generated 3 negatives
2026-03-27 05:21:19 | INFO     | pepbenchmark | Testing NN

/home/dataset-assist-0/jiahui/pepbenchmark/final/pepbenchmark/src/pepbenchmark/neg_sampler/sampling_strategies.py:925: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  counts = pos_bin_df.groupby(keys).size().sort_index()


## Summary

- Use `SamplingPoolManager` to manage the candidate pool
- Use `NegSampler` to run sampling
- Use `generate_similarity_report()` / `check_similarity()` for quality assessment
- Use `StrategySelector` to compare multiple strategies